In [1]:
import geopandas as gpd
from pathlib import Path
import pandas as pd

folder = Path(r"C:\git\Master-Thesis-GEOAI\data\spacenet_khartoum\AOI_5_Khartoum_Train\geojson\buildings")

files = list(folder.glob("*.geojson"))

print(f"Found {len(files)} files")

gdfs = [gpd.read_file(f) for f in files]

gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

out = r"C:\git\Master-Thesis-GEOAI\data\spacenet_khartoum\spacenet_buildings.geojson"

gdf.to_file(out, driver="GeoJSON")

print("Saved:", out)


Found 1012 files
Saved: C:\git\Master-Thesis-GEOAI\data\spacenet_khartoum\spacenet_buildings.geojson


In [2]:
import rasterio

with rasterio.open(r"C:\temp\65a5b537c2e5aa0001f0a1f4.tif") as ds:
    print(ds.crs)
    print(ds.transform)
    print(ds.res)

PROJCS["WGS 84 / UTM zone 42N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",69],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
| 0.31, 0.00, 539843.75|
| 0.00,-0.31, 3690156.25|
| 0.00, 0.00, 1.00|
(0.30517578125000006, 0.30517578125000006)


In [6]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

with rasterio.open(r"C:\temp\65a5b537c2e5aa0001f0a1f4.tif") as ds:
    transform, width, height = calculate_default_transform(
        ds.crs,
        "EPSG:32642",
        ds.width,
        ds.height,
        *ds.bounds,
        resolution=0.30
    )

print(transform, width, height)


| 0.30, 0.00, 539843.75|
| 0.00,-0.30, 3690156.25|
| 0.00, 0.00, 1.00| 67709 167709


In [8]:
from transformers import Sam3Model, Sam3Processor
import torch

SAM_PATH = "C:\git\Master-Thesis-GEOAI\models\sam3_weights"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam_model = Sam3Model.from_pretrained(SAM_PATH).to(device)
sam_processor = Sam3Processor.from_pretrained(SAM_PATH)

print("SAM3 loaded on", device)

Loading weights: 100%|██████████| 1468/1468 [00:02<00:00, 694.26it/s, Materializing param=vision_encoder.neck.fpn_layers.3.proj2.weight]                       


SAM3 loaded on cuda


In [1]:
print("hello world")

hello world


In [19]:
import geopandas as gpd
from sqlalchemy import create_engine, text
from geopy.geocoders import Nominatim
import os
import re

AOI_PATH = r"C:\Users\franz\Documents\ArcGIS\Projects\GeoAI_TestData\GeoAI_TestData.gdb"
AOI_LAYER = "aoi"
PG_CONN = os.environ["PG_CONN"]

engine = create_engine(PG_CONN)

# =====================================
# Load AOIs
# =====================================

aoi = gpd.read_file(AOI_PATH, layer=AOI_LAYER)

print("Loaded:", len(aoi))

# WGS84
aoi = aoi.to_crs(4326)

aoi = aoi[["geometry"]].copy()
aoi = aoi.rename_geometry("geom")

# =====================================
# Reverse geocoding
# =====================================

geolocator = Nominatim(user_agent="geoai")

names = []

for geom in aoi.geom:

    lon, lat = geom.centroid.xy

    try:
        loc = geolocator.reverse(f"{lat[0]}, {lon[0]}", zoom=10)
    except Exception as e:
        print("Geocode failed:", e)
        loc = None

    if loc:
        addr = loc.raw["address"]

        city = (
            addr.get("city")
            or addr.get("town")
            or addr.get("village")
            or addr.get("state")
            or "unknown"
        )

        country = addr.get("country") or "unknown"

        print("Detected:", city, country)

        name = f"{city}_{country}"
    else:
        name = "unknown"

    name = re.sub(r"\s+", "_", name.lower())
    name = re.sub(r"[^a-z0-9_]", "", name)

    names.append(name)

aoi["name"] = names

# =====================================
# Proper area (meters² via UTM)
# =====================================

aoi_m = aoi.to_crs(aoi.estimate_utm_crs())
aoi["area_m2"] = aoi_m.area

# IDs
aoi["aoi_id"] = range(1, len(aoi) + 1)

print(aoi[["aoi_id", "name", "area_m2"]])

# =====================================
# Recreate table
# =====================================

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS src.aoi CASCADE;"))

# =====================================
# Write DB
# =====================================

aoi.to_postgis(
    name="aoi",
    con=engine,
    schema="src",
    if_exists="replace",
    index=False
)

with engine.begin() as conn:

    conn.execute(text("""
        ALTER TABLE src.aoi
        ADD PRIMARY KEY (aoi_id);
    """))

    conn.execute(text("""
        CREATE INDEX aoi_geom_idx
        ON src.aoi
        USING GIST (geom);
    """))

# =====================================
# Verify
# =====================================

check = gpd.read_postgis(
    "SELECT aoi_id,name,area_m2, geom FROM src.aoi",
    engine,
    geom_col="geom"
)


print("\nAOIs in DB:")
print(check)

print("\nDONE ✅")


Loaded: 3
Detected: Monrovia Liberia
Detected: Ouagadougou Burkina Faso
Detected: Niamey Niger
   aoi_id                      name       area_m2
0       1          monrovia_liberia  1.513202e+05
1       2  ouagadougou_burkina_faso  5.747977e+05
2       3              niamey_niger  1.253433e+06

AOIs in DB:
   aoi_id                      name       area_m2  \
0       1          monrovia_liberia  1.513202e+05   
1       2  ouagadougou_burkina_faso  5.747977e+05   
2       3              niamey_niger  1.253433e+06   

                                                geom  
0  MULTIPOLYGON Z (((-10.79097 6.33414 0, -10.788...  
1  MULTIPOLYGON Z (((-1.60151 12.28761 0, -1.5946...  
2  MULTIPOLYGON Z (((2.13347 13.47469 0, 2.13668 ...  

DONE ✅
